In [ ]:
# Install google sheets libraries
!pip install gspread pandas gspread_dataframe

In [ ]:
# Authenticate google account in colab
from google.colab import auth
auth.authenticate_user()

# Connect to google sheets
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

# Authenticate and create client
creds, _ = default()
gc = gspread.authorize(creds)

# Create new google sheets
spreadsheet = gc.create("IT Ticket Report")

# Share email
spreadsheet.share('insertemail@gmail.com', perm_type='user', role='writer')

<Response [200]>

In [ ]:
# Sqlite setup
# Import sqlite libraries
import sqlite3
import pandas as pd

# Create a new sqlite database in memory
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

**Database Schema**

In [ ]:
# Departments table
cursor.execute("""
CREATE TABLE Departments (
    DepartmentID INTEGER PRIMARY KEY,
    DepartmentName TEXT
)
""")

| Column Name    | Data Type | Description            |
| -------------- | --------- | ---------------------- |
| DepartmentID   | INTEGER   | Primary Key            |
| DepartmentName | TEXT      | Name of the department |


In [ ]:
# Rootcauses table
cursor.execute("""
CREATE TABLE RootCauses (
    RootCauseID INTEGER PRIMARY KEY,
    RootCause TEXT
)
""")

| Column Name | Data Type | Description           |
| ----------- | --------- | --------------------- |
| RootCauseID | INTEGER   | Primary Key           |
| RootCause   | TEXT      | Reason for the ticket |


In [ ]:
# IT_Tickets table
cursor.execute("""
CREATE TABLE IT_Tickets (
    TicketID INTEGER PRIMARY KEY,
    DepartmentID INTEGER,
    RootCauseID INTEGER,
    Status TEXT,
    Priority TEXT,
    ResolutionHours INTEGER,
    CreatedDate TEXT,
    FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID),
    FOREIGN KEY (RootCauseID) REFERENCES RootCauses(RootCauseID)
)
""")

| Column Name     | Data Type | Description                         |
| --------------- | --------- | ----------------------------------- |
| TicketID        | INTEGER   | Primary Key                         |
| DepartmentID    | INTEGER   | Foreign Key referencing Departments |
| RootCauseID     | INTEGER   | Foreign Key referencing RootCauses  |
| Status          | TEXT      | Ticket status: Open, Closed         |
| Priority        | TEXT      | Priority level: High, Medium, Low   |
| ResolutionHours | INTEGER   | Hours taken to resolve the ticket   |
| CreatedDate     | TEXT      | Ticket creation date (YYYY-MM-DD)   |


**Insert data into table**

In [ ]:
# Insert data into Departments
departments = [
    (1,'IT'),
    (2,'HR'),
    (3,'Finance'),
    (4,'Operations'),
    (5,'Customer Service'),

]

conn.executemany("INSERT INTO Departments VALUES (?,?)", departments)
conn.commit()

# Verify
import pandas as pd
pd.read_sql("SELECT * FROM Departments", conn)

,DepartmentID,DepartmentName
0,1,IT
1,2,HR
2,3,Finance
3,4,Operations
4,5,Customer Service


In [ ]:
# Insert data into RootCauses
root_causes = [
    (1,'Configuration'),
    (2,'Software Failure'),
    (3,'Network Issue'),
    (4,'Hardware Failure'),
    (5,'Security')
]

conn.executemany("INSERT INTO RootCauses VALUES (?,?)", root_causes)
conn.commit()

# Verify
pd.read_sql("SELECT * FROM RootCauses", conn)

,RootCauseID,RootCause
0,1,Configuration
1,2,Software Failure
2,3,Network Issue
3,4,Hardware Failure
4,5,Security


In [ ]:
# Insert data into IT_Tickets
tickets = [
    (1,1,1,'Closed','High',5,'2026-03-01'),
    (2,2,5,'Closed','Medium',8,'2026-03-02'),
    (3,1,3,'Open','Low',12,'2026-03-03'),
    (4,5,1,'Closed','High',3,'2026-03-04'),
    (5,4,4,'Open','Medium',7,'2026-03-05'),
    (6,2,2,'Closed','Low',4,'2026-03-06'),
    (7,3,3,'Closed','High',6,'2026-03-07'),
    (8,1,1,'Open','High',9,'2026-03-08'),
    (9,4,2,'Closed','Medium',2,'2026-03-09'),
    (10,5,4,'Open','Low',10,'2026-03-10'),
    (11,5,2,'Closed','Low',4,'2026-01-04'),
    (12,3,3,'Closed','High',6,'2026-01-11'),
    (13,1,1,'Open','Low',9,'2026-03-01'),
    (14,4,5,'Closed','Medium',2,'2026-01-03'),
    (15,5,4,'Open','Low',10,'2026-01-11')
]

conn.executemany("INSERT INTO IT_Tickets VALUES (?,?,?,?,?,?,?)", tickets)
conn.commit()

# Verify
pd.read_sql("SELECT * FROM IT_Tickets", conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,1,1,1,Closed,High,5,2026-03-01
1,2,2,5,Closed,Medium,8,2026-03-02
2,3,1,3,Open,Low,12,2026-03-03
3,4,5,1,Closed,High,3,2026-03-04
4,5,4,4,Open,Medium,7,2026-03-05
5,6,2,2,Closed,Low,4,2026-03-06
6,7,3,3,Closed,High,6,2026-03-07
7,8,1,1,Open,High,9,2026-03-08
8,9,4,2,Closed,Medium,2,2026-03-09
9,10,5,4,Open,Low,10,2026-03-10


# **Extract**


In [ ]:
# Extract data from sqlite
query = """
SELECT *
FROM IT_Tickets
"""
df = pd.read_sql(query, conn)
df.head()

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,1,1,1,Closed,High,5,2026-03-01
1,2,2,5,Closed,Medium,8,2026-03-02
2,3,1,3,Open,Low,12,2026-03-03
3,4,5,1,Closed,High,3,2026-03-04
4,5,4,4,Open,Medium,7,2026-03-05


# **Transform**

In [ ]:
# Transform ticket by department
# Group by DepartmentID and TicketID then count
ticketsDepartment = df.groupby('DepartmentID')['TicketID'].count().reset_index()

# Rename column
ticketsDepartment.rename(columns={'TicketID': 'Total_Tickets'}, inplace=True)

# Print table
ticketsDepartment

,DepartmentID,Total_Tickets
0,1,4
1,2,2
2,3,2
3,4,3
4,5,4


In [ ]:
# Tickets by Status & Priority
# Group by Status and Priority then count
ticketsPriorityStatus = df.groupby(['Status', 'Priority'])['TicketID'].count().reset_index()

# Rename column
ticketsPriorityStatus.rename(columns={'TicketID': 'Count'}, inplace=True)

# Print table
ticketsPriorityStatus

,Status,Priority,Count
0,Closed,High,4
1,Closed,Low,2
2,Closed,Medium,3
3,Open,High,1
4,Open,Low,4
5,Open,Medium,1


In [ ]:
# Average Resolution Time
# Convert date to datetime
df['CreatedDate'] = pd.to_datetime(df['CreatedDate'])

# Calculate average resolution time (hours) per department
averageResolution = df.groupby('DepartmentID')['ResolutionHours'].mean().reset_index()

# Round the average resolution hours to 2 decimal places
averageResolution['ResolutionHours'] = averageResolution['ResolutionHours'].round(2)

# Rename column
averageResolution.rename(columns={'ResolutionHours': 'Average_Resolution_Hours'}, inplace=True)

# Print table
averageResolution

,DepartmentID,Average_Resolution_Hours
0,1,8.75
1,2,6.00
2,3,6.00
3,4,3.67
4,5,6.75


# **Load**

In [ ]:
# Load tickets data into google sheets
# Department
worksheet1 = spreadsheet.get_worksheet(0)
worksheet1.update_title("Department")
set_with_dataframe(worksheet1, ticketsDepartment)

# Status and Priority
worksheet2 = spreadsheet.add_worksheet(title="Status & Priority", rows="100", cols="20")
set_with_dataframe(worksheet2, ticketsPriorityStatus)

# Average Resolution Time
worksheet3 = spreadsheet.add_worksheet(title="Average Resolution Time", rows="100", cols="20")
set_with_dataframe(worksheet3, averageResolution)

In [ ]:
# Open google sheets
print("Google Sheet URL:", spreadsheet.url)